In [3]:
#!/usr/bin/env python3
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


# -------------------------
# IMPORTANT: pickle compatibility shim
# -------------------------
class _LBFGSTimer:
    """
    Only here so joblib can unpickle models trained with optimizer=_LBFGSTimer()
    defined in __main__. It will not be called for prediction.
    """
    def __init__(self):
        self.restart_counter = 0

    def __call__(self, obj_func, initial_theta, bounds):
        raise RuntimeError(
            "_LBFGSTimer was invoked. You're probably refitting; this script is prediction-only."
        )


# -------------------------
# CONFIG
# -------------------------
csv_path = "/Users/justint/Library/CloudStorage/OneDrive-Personal/Desktop/Academic Stuff/Arias Research/Polymers/2025/Paper_Polymers_2025/new/SCFT/triblock_ML_dataset/dataset.csv"

models_dir = "/Users/justint/Library/CloudStorage/OneDrive-Personal/Desktop/Academic Stuff/Arias Research/Polymers/2025/Paper_Polymers_2025/new/SCFT/phase_visualizer/models/ABC"

out_dir = "/Users/justint/Library/CloudStorage/OneDrive-Personal/Desktop/Academic Stuff/Arias Research/Polymers/2025/Paper_Polymers_2025/new/SCFT/triblock_ML_dataset/plots_eval"
os.makedirs(out_dir, exist_ok=True)

feature_cols = ["chiAB", "chiAC", "chiBC", "fA", "fC"]
morphologies = ["cylinders", "spheres", "gyroid", "lamella", "O70", "diamond"]

rng_state = 88
test_size = 0.10


# -------------------------
# Load dataset + split
# -------------------------
df = pd.read_csv(csv_path)

all_rows = np.arange(len(df))
rows_tr, rows_te = train_test_split(all_rows, test_size=test_size, random_state=rng_state)

row_tr_mask = np.zeros(len(df), dtype=bool)
row_tr_mask[rows_tr] = True
row_te_mask = ~row_tr_mask


# -------------------------
# Load trained GPs
# -------------------------
gp_models = {}
missing = []
for morph in morphologies:
    pkl = os.path.join(models_dir, f"gp_{morph}.pkl")
    if os.path.exists(pkl):
        gp_models[morph] = joblib.load(pkl)
    else:
        missing.append(morph)

if missing:
    print("WARNING: missing model files for:", missing)
if not gp_models:
    raise RuntimeError("No GP models found. Check models_dir/csv_path.")


# -------------------------
# FE MAE table helper
# -------------------------
def fe_mae_table(mask, label):
    rows = []
    for morph, gp in gp_models.items():
        col = f"FE_{morph}"
        valid = mask & df[col].notna()
        X = df.loc[valid, feature_cols].values
        y = df.loc[valid, col].values

        mae = mean_absolute_error(y, gp.predict(X)) if len(y) else np.nan
        rows.append({"morphology": morph, "n": len(y), f"mae_{label}": mae})

    out = pd.DataFrame(rows).sort_values(f"mae_{label}", na_position="last")
    print(f"\nFE MAE ({label.upper()})")
    print(out.to_string(index=False))
    out_path = os.path.join(out_dir, f"fe_mae_{label}.csv")
    out.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")
    return out


# -------------------------
# ΔF helpers
# -------------------------
def delta_pair(mask, m1, m2):
    if (m1 not in gp_models) or (m2 not in gp_models):
        return np.array([]), np.array([])

    col1, col2 = f"FE_{m1}", f"FE_{m2}"
    valid = mask & df[col1].notna() & df[col2].notna()
    if not np.any(valid):
        return np.array([]), np.array([])

    X = df.loc[valid, feature_cols].values
    a = df.loc[valid, col1].values - df.loc[valid, col2].values
    p = gp_models[m1].predict(X) - gp_models[m2].predict(X)
    return a, p


def gather(mask):
    A, P = [], []
    for i, j in combinations(range(len(morphologies)), 2):
        m1, m2 = morphologies[i], morphologies[j]
        a, p = delta_pair(mask, m1, m2)
        if len(a):
            A.append(a)
            P.append(p)
    if not A:
        return np.array([]), np.array([])
    return np.concatenate(A), np.concatenate(P)


def mae_matrix(mask):
    n = len(morphologies)
    M = np.full((n, n), np.nan, dtype=float)
    for i, j in combinations(range(n), 2):
        m1, m2 = morphologies[i], morphologies[j]
        a, p = delta_pair(mask, m1, m2)
        if len(a):
            mae = mean_absolute_error(a, p)
            M[i, j] = M[j, i] = mae
    np.fill_diagonal(M, 0.0)
    return M


def save_matrix_txt(M, label):
    out_path = os.path.join(out_dir, f"mae_matrix_{label}.txt")
    header = [""] + morphologies
    widths = [max(10, max(len(h) for h in header))] + [max(10, len(m)) for m in morphologies]

    def fmt_row(cells):
        return " ".join(str(c).rjust(w) for c, w in zip(cells, widths))

    lines = [f"— {label.upper()} ΔF MAE matrix —", fmt_row(header)]
    for i, mi in enumerate(morphologies):
        row_vals = [(f"{M[i,j]:.4f}" if np.isfinite(M[i,j]) else "nan") for j in range(len(morphologies))]
        lines.append(fmt_row([mi] + row_vals))

    with open(out_path, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"Saved -> {out_path}")


def pooled_mae(a, p):
    return mean_absolute_error(a, p) if len(a) else np.nan


# -------------------------
# Run everything
# -------------------------
train_fe = fe_mae_table(row_tr_mask, "train")
test_fe  = fe_mae_table(row_te_mask, "test")

a_tr, p_tr = gather(row_tr_mask)
a_te, p_te = gather(row_te_mask)

M_tr = mae_matrix(row_tr_mask)
M_te = mae_matrix(row_te_mask)
save_matrix_txt(M_tr, "train")
save_matrix_txt(M_te, "test")

print("\nOverall pooled ΔF MAE")
print("TRAIN:", pooled_mae(a_tr, p_tr))
print("TEST :", pooled_mae(a_te, p_te))
print("\nDone.")



FE MAE (TRAIN)
morphology    n  mae_train
   diamond 5165   0.007190
    gyroid 6174   0.010352
   lamella 6840   0.011377
       O70 4793   0.013156
   spheres 4569   0.015176
 cylinders 6500   0.016739
Saved -> /Users/justint/Library/CloudStorage/OneDrive-Personal/Desktop/Academic Stuff/Arias Research/Polymers/2025/Paper_Polymers_2025/new/SCFT/triblock_ML_dataset/plots_eval/fe_mae_train.csv

FE MAE (TEST)
morphology   n  mae_test
   diamond 603  0.009256
    gyroid 688  0.012026
       O70 534  0.015804
   spheres 505  0.019572
 cylinders 732  0.021250
   lamella 768  0.022131
Saved -> /Users/justint/Library/CloudStorage/OneDrive-Personal/Desktop/Academic Stuff/Arias Research/Polymers/2025/Paper_Polymers_2025/new/SCFT/triblock_ML_dataset/plots_eval/fe_mae_test.csv
Saved -> /Users/justint/Library/CloudStorage/OneDrive-Personal/Desktop/Academic Stuff/Arias Research/Polymers/2025/Paper_Polymers_2025/new/SCFT/triblock_ML_dataset/plots_eval/mae_matrix_train.txt
Saved -> /Users/justint/Li

In [56]:
def single_plot(a, p, label, filename, split_name):
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 10))

    if len(a) == 0:
        ax.text(0.5, 0.5, "no data",
                ha="center", va="center", transform=ax.transAxes)
    else:
        # ---- Robust axis limits (ignore extreme outliers for view) ----
        data = np.concatenate([a, p])
        lo = np.percentile(data, 0.00)
        hi = np.percentile(data, 1.0)

        # Fallback if data is (nearly) constant or percentiles collapse
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo = float(np.min(data))
            hi = float(np.max(data))
            if hi <= lo:
                hi = lo + 1.0

        buf = 0.02 * (hi - lo)
        mn = lo - buf
        mx = hi + buf

        low, high = -1.0, 1.0
        ax.plot([low, high], [low, high], "k--", linewidth=1.0, color='r')
        ax.scatter(a, p, s=30, edgecolor="b", linewidth=0.0, alpha=0.03, color='b')
        ax.set_xlim(low, high)
        ax.set_ylim(low, high)
        ax.set_aspect("equal", "box")

        # Use mathtext-safe LaTeX (avoid \text{} unless usetex=True)
        ax.set_xlabel(r"$\mathrm{Actual}\ \Delta F$", fontsize=20)
        ax.set_ylabel(r"$\mathrm{Predicted}\ \Delta F$", fontsize=20)
        ax.tick_params(labelsize=18)

        mae = pooled_mae(a, p)

        ax.text(
            0.05, 0.95,
            f"MAE ({split_name}) = {mae:.4f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=18,
            bbox=dict(facecolor="white", alpha=1.0, edgecolor="none")
        )

    fig.tight_layout()

    out = os.path.join(out_dir, filename)
    fig.savefig(out, bbox_inches="tight", dpi=300)
    plt.close(fig)

    print(f"Saved -> {out}")


single_plot(a_tr, p_tr, "TRAIN ΔF", "deltaF_train.png", "training")
single_plot(a_te, p_te, "TEST ΔF",  "deltaF_test.png",  "testing")


/var/folders/yl/sv3dx7zs13s3phj74920ys6w0000gn/T/ipykernel_87775/3346577473.py:29: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "k--" (-> color='k'). The keyword argument will take precedence.
  ax.plot([low, high], [low, high], "k--", linewidth=1.0, color='r')


Saved -> /Users/justint/Library/CloudStorage/OneDrive-Personal/Desktop/Academic Stuff/Arias Research/Polymers/2025/Paper_Polymers_2025/new/SCFT/triblock_ML_dataset/plots_eval/deltaF_train.png
Saved -> /Users/justint/Library/CloudStorage/OneDrive-Personal/Desktop/Academic Stuff/Arias Research/Polymers/2025/Paper_Polymers_2025/new/SCFT/triblock_ML_dataset/plots_eval/deltaF_test.png


/var/folders/yl/sv3dx7zs13s3phj74920ys6w0000gn/T/ipykernel_87775/3346577473.py:29: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "k--" (-> color='k'). The keyword argument will take precedence.
  ax.plot([low, high], [low, high], "k--", linewidth=1.0, color='r')
